<a href="https://colab.research.google.com/github/MO230101/Copolymer-lipid-interaction-study_ver.2/blob/main/%E3%80%87M_I_Caption_Feature_Generator_for_ROI_Resolved_NMR_Profiles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===============================================================
# Google Colab COMPLETE CODE
# Integrated M + I caption-feature CSV generator
#
# Integrates:
#   code3: change-based motif scoring from Blank-relative ROI features
#   code1: paper-ready interaction profile reconstruction
#
# Main purpose:
#   Generate CSV containing M and I information for caption generation.
#
# Required input files:
#   1) roi_peak_features_with_blank_relative.csv
#   2) ZG_to_FunctionalGroup_weights.csv
#
# Optional input:
#   3) Copolymer_ROI_motif_scores_260312.csv
#      If present, original intensity-based motif scores are merged.
#
# Main outputs:
#   - Copolymer_change_based_motif_scores.csv
#   - Copolymer_MI_caption_features.csv
#   - Copolymer_MI_caption_features.xlsx
#   - Copolymer_MI_caption_summary.csv
#   - Copolymer_MI_caption_summary.xlsx
# ===============================================================

!pip -q install pandas numpy openpyxl xlsxwriter

import os
import re
import glob
import numpy as np
import pandas as pd

from google.colab import files

# ===============================================================
# 0. Upload files if needed
# ===============================================================
def maybe_upload():
    need_roi = len(glob.glob("/content/*roi_peak_features_with_blank_relative*.csv")) == 0
    need_w   = len(glob.glob("/content/*ZG_to_FunctionalGroup_weights*.csv")) == 0

    if need_roi or need_w:
        print("Please upload required files:")
        print(" - roi_peak_features_with_blank_relative.csv")
        print(" - ZG_to_FunctionalGroup_weights.csv")
        print("Optional:")
        print(" - Copolymer_ROI_motif_scores_260312.csv")
        files.upload()

maybe_upload()

# ===============================================================
# 1. File finder
# ===============================================================
def find_one(patterns):
    hits = []
    for p in patterns:
        hits.extend(glob.glob(p))
    hits = sorted(set(hits))
    return hits[0] if hits else None

ROI_FEATURE_PATH = find_one([
    "/content/roi_peak_features_with_blank_relative.csv",
    "/content/*roi_peak_features_with_blank_relative*.csv",
])

WEIGHT_PATH = find_one([
    "/content/ZG_to_FunctionalGroup_weights.csv",
    "/content/*ZG_to_FunctionalGroup_weights*.csv",
])

BASE_MOTIF_PATH = find_one([
    "/content/Copolymer_ROI_motif_scores_260312.csv",
    "/content/*Copolymer_ROI_motif_scores_260312*.csv",
])

if ROI_FEATURE_PATH is None:
    raise FileNotFoundError("roi_peak_features_with_blank_relative.csv not found.")
if WEIGHT_PATH is None:
    raise FileNotFoundError("ZG_to_FunctionalGroup_weights.csv not found.")

print("ROI_FEATURE_PATH:", ROI_FEATURE_PATH)
print("WEIGHT_PATH     :", WEIGHT_PATH)
print("BASE_MOTIF_PATH :", BASE_MOTIF_PATH)

# ===============================================================
# 2. Readers
# ===============================================================
def robust_read_csv(path):
    last_err = None
    for enc in ["utf-8-sig", "utf-8", "cp932", "latin1"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"[INFO] Loaded {os.path.basename(path)} with encoding: {enc}")
            return df
        except Exception as e:
            last_err = e
    raise last_err

roi_df = robust_read_csv(ROI_FEATURE_PATH)
w_df   = robust_read_csv(WEIGHT_PATH)

roi_df.columns = [str(c).strip() for c in roi_df.columns]
w_df.columns   = [str(c).strip() for c in w_df.columns]

base_df = None
if BASE_MOTIF_PATH is not None:
    base_df = robust_read_csv(BASE_MOTIF_PATH)
    base_df.columns = [str(c).strip() for c in base_df.columns]

print("\nROI feature columns:")
print(roi_df.columns.tolist())
print("\nWeight columns:")
print(w_df.columns.tolist())
if base_df is not None:
    print("\nBase motif columns:")
    print(base_df.columns.tolist())

# ===============================================================
# 3. Validation
# ===============================================================
required_roi_cols = ["No", "Sample_Name", "ROI_Label", "delta_height", "delta_auc"]
missing_roi = [c for c in required_roi_cols if c not in roi_df.columns]
if missing_roi:
    raise ValueError(f"roi_peak_features_with_blank_relative.csv is missing columns: {missing_roi}")

required_w_cols = ["ZG_ROI", "FunctionalGroup", "weight"]
missing_w = [c for c in required_w_cols if c not in w_df.columns]
if missing_w:
    raise ValueError(f"ZG_to_FunctionalGroup_weights.csv is missing columns: {missing_w}")

# ===============================================================
# 4. Normalization utilities
# ===============================================================
MOTIFS = ["headgroup", "glycerol", "alkenyl", "alkyl_chain"]

TARGET_GROUPS = {
    "headgroup":   "headgroup_polar_ch",
    "glycerol":    "glycerol_oxygenated_ch",
    "alkenyl":     "alkenyl_ch",
    "alkyl_chain": "alkyl_chain_chx",
}

def normalize_roi_name(x):
    s = str(x).strip()
    s = s.replace("ROI ", "ROI.")
    s = s.replace("ROI_", "ROI.")
    s = s.replace("ROI-", "ROI.")
    m = re.search(r"ROI\.?(\d+)", s, flags=re.IGNORECASE)
    if m:
        return f"ROI.{m.group(1)}"
    return s

def normalize_fg_name(x):
    s = str(x).strip().lower()
    s = s.replace("/", "_")
    s = s.replace("-", "_")
    s = re.sub(r"__+", "_", s).strip("_")
    return s

def normalize_copolymer_name(x):
    s = str(x).strip()
    s = s.replace("4VBA", "VBA")
    s = s.replace("MEDSH", "MEDSAH")
    s = s.replace("TECL2", "TECL")
    s = s.replace("-", "_")
    s = re.sub(r"\s+", "", s)
    return s

def parse_copolymer_name(name):
    parts = normalize_copolymer_name(name).split("_")
    if len(parts) < 3:
        return (np.nan, np.nan, np.nan)
    return parts[0], parts[1], parts[2]

def roi_sort_key(x):
    m = re.search(r"(\d+)", str(x))
    return int(m.group(1)) if m else 999

def safe_float(x):
    try:
        if pd.isna(x):
            return np.nan
        return float(x)
    except Exception:
        return np.nan

# ===============================================================
# 5. Preprocess input tables
# ===============================================================
roi_df["ROI_Label"] = roi_df["ROI_Label"].apply(normalize_roi_name)
w_df["ZG_ROI"] = w_df["ZG_ROI"].apply(normalize_roi_name)
w_df["FunctionalGroup"] = w_df["FunctionalGroup"].apply(normalize_fg_name)
w_df["weight"] = pd.to_numeric(w_df["weight"], errors="coerce").fillna(0.0)

# Remove Blank row if No == 0
roi_df = roi_df[pd.to_numeric(roi_df["No"], errors="coerce") != 0].copy()

# delta = sample - blank
# stronger decrease relative to Blank is negative delta.
# suppression = -delta, clipped at zero.
roi_df["suppression_height"] = -pd.to_numeric(roi_df["delta_height"], errors="coerce")
roi_df["suppression_auc"]    = -pd.to_numeric(roi_df["delta_auc"], errors="coerce")

roi_df["suppression_height"] = roi_df["suppression_height"].where(
    roi_df["suppression_height"].isna(),
    roi_df["suppression_height"].clip(lower=0)
)
roi_df["suppression_auc"] = roi_df["suppression_auc"].where(
    roi_df["suppression_auc"].isna(),
    roi_df["suppression_auc"].clip(lower=0)
)

roi_set = set(roi_df["ROI_Label"].dropna().unique())
w_set   = set(w_df["ZG_ROI"].dropna().unique())
common_rois = sorted(roi_set & w_set, key=roi_sort_key)

print("\n[INFO] Common ROIs used:", common_rois)
if len(common_rois) == 0:
    raise ValueError("No overlapping ROI names found between ROI feature table and weight table.")

roi_df = roi_df[roi_df["ROI_Label"].isin(common_rois)].copy()
w_df   = w_df[w_df["ZG_ROI"].isin(common_rois)].copy()

# ===============================================================
# 6. Build functional-group weights
# ===============================================================
w_piv = w_df.pivot_table(
    index="ZG_ROI",
    columns="FunctionalGroup",
    values="weight",
    aggfunc="sum",
    fill_value=0.0
).reset_index()

weights_by_group = {}
for motif, fg_name in TARGET_GROUPS.items():
    if fg_name in w_piv.columns:
        weights_by_group[motif] = dict(zip(w_piv["ZG_ROI"], w_piv[fg_name]))
    else:
        print(f"[WARN] {fg_name} not found. Filling zeros for {motif}.")
        weights_by_group[motif] = {roi: 0.0 for roi in common_rois}

# ===============================================================
# 7. code3 part: change-based motif scores
# ===============================================================
agg_roi = roi_df.groupby(
    ["No", "Sample_Name", "ROI_Label"],
    as_index=False
)[["suppression_height", "suppression_auc"]].mean()

sample_keys = agg_roi[["No", "Sample_Name"]].drop_duplicates().copy()
sample_keys["Copolymer_Name"] = sample_keys["Sample_Name"].apply(normalize_copolymer_name)

result_rows = []

for _, srow in sample_keys.iterrows():
    no = srow["No"]
    sname = srow["Sample_Name"]
    cop = srow["Copolymer_Name"]

    sub = agg_roi[(agg_roi["No"] == no) & (agg_roi["Sample_Name"] == sname)].copy()

    sub_dict_h = dict(zip(sub["ROI_Label"], sub["suppression_height"]))
    sub_dict_a = dict(zip(sub["ROI_Label"], sub["suppression_auc"]))

    rec = {
        "No": no,
        "Sample_Name": sname,
        "Copolymer_Name": cop,
    }

    for motif in MOTIFS:
        score_h = 0.0
        score_a = 0.0

        for roi in common_rois:
            w = weights_by_group[motif].get(roi, 0.0)
            v_h = sub_dict_h.get(roi, np.nan)
            v_a = sub_dict_a.get(roi, np.nan)

            if pd.notna(v_h):
                score_h += float(v_h) * float(w)
            if pd.notna(v_a):
                score_a += float(v_a) * float(w)

        rec[f"{motif}_delta_height_score"] = score_h
        rec[f"{motif}_delta_auc_score"] = score_a

    result_rows.append(rec)

motif_df = pd.DataFrame(result_rows)

height_cols = [f"{m}_delta_height_score" for m in MOTIFS]
auc_cols    = [f"{m}_delta_auc_score" for m in MOTIFS]

motif_df["total_delta_height_score"] = motif_df[height_cols].sum(axis=1)
motif_df["total_delta_auc_score"]    = motif_df[auc_cols].sum(axis=1)

for c in height_cols:
    motif_df[c.replace("_score", "_norm")] = (
        motif_df[c] / motif_df["total_delta_height_score"].replace(0, np.nan)
    )

for c in auc_cols:
    motif_df[c.replace("_score", "_norm")] = (
        motif_df[c] / motif_df["total_delta_auc_score"].replace(0, np.nan)
    )

def dominant_from_cols(row, cols, suffix_to_strip):
    vals = row[cols].values.astype(float)
    if np.all(np.isnan(vals)):
        return "unknown"
    idx = int(np.nanargmax(vals))
    return cols[idx].replace(suffix_to_strip, "")

motif_df["dominant_height_motif"] = motif_df.apply(
    lambda r: dominant_from_cols(r, height_cols, "_delta_height_score"),
    axis=1
)
motif_df["dominant_auc_motif"] = motif_df.apply(
    lambda r: dominant_from_cols(r, auc_cols, "_delta_auc_score"),
    axis=1
)

parsed = motif_df["Copolymer_Name"].apply(parse_copolymer_name)
motif_df["hydrophilic_monomer"] = parsed.apply(lambda x: x[0])
motif_df["hydrophobic_monomer"] = parsed.apply(lambda x: x[1])
motif_df["crosslinker"] = parsed.apply(lambda x: x[2])

# Save code3-equivalent output
CHANGE_CSV  = "/content/Copolymer_change_based_motif_scores.csv"
CHANGE_XLSX = "/content/Copolymer_change_based_motif_scores.xlsx"

motif_df.to_csv(CHANGE_CSV, index=False, encoding="utf-8-sig")
with pd.ExcelWriter(CHANGE_XLSX, engine="xlsxwriter") as writer:
    motif_df.to_excel(writer, index=False, sheet_name="change_based_scores")

print("\nSaved change-based motif scores:")
print(" -", CHANGE_CSV)
print(" -", CHANGE_XLSX)

# ===============================================================
# 8. code1 part: interaction profile reconstruction
# ===============================================================
SETTINGS = {
    "SOFTMAX_TEMPERATURE_HEIGHT": 0.60,
    "SOFTMAX_TEMPERATURE_AUC": 0.60,

    "SECOND_MIN_SOFT": 0.18,
    "SECOND_MIN_RAW_RATIO": 0.10,
    "SECOND_MIN_REL_TO_TOP1": 0.30,

    "STRONG_SOFT_MARGIN": 0.30,
    "MODERATE_SOFT_MARGIN": 0.18,

    "LOCALIZED_ENTROPY_MAX": 0.45,
    "DUAL_ENTROPY_MAX": 0.72,

    "AMBIGUOUS_TOP2_GAP": 0.08,
}

def softmax_temperature(x, T=0.6):
    arr = np.array(x, dtype=float)
    arr = np.where(np.isfinite(arr), arr, np.nan)

    if np.all(np.isnan(arr)):
        return np.full(len(arr), np.nan)

    arr = np.where(np.isnan(arr), -np.inf, arr)
    finite = arr[np.isfinite(arr)]
    if len(finite) == 0:
        return np.full(len(arr), np.nan)

    m = np.max(finite)
    z = (arr - m) / max(T, 1e-6)

    expz = np.where(np.isfinite(z), np.exp(z), 0.0)
    denom = np.sum(expz)

    if denom <= 0:
        return np.full(len(arr), np.nan)

    return expz / denom

def raw_to_ratio(raw):
    raw = np.array(raw, dtype=float)
    if np.all(~np.isfinite(raw)):
        return np.full(len(raw), np.nan)

    raw = np.where(np.isfinite(raw), raw, 0.0)
    raw = np.clip(raw, 0, None)

    s = raw.sum()
    if s <= 0:
        return np.full(len(raw), np.nan)

    return raw / s

def normalized_entropy(probs):
    arr = np.array(probs, dtype=float)
    arr = arr[np.isfinite(arr)]
    arr = arr[arr > 0]

    if len(arr) == 0:
        return np.nan

    h = -np.sum(arr * np.log(arr))
    hmax = np.log(len(MOTIFS))

    if hmax <= 0:
        return 0.0

    return float(h / hmax)

def rank_motifs(values):
    pairs = list(zip(MOTIFS, values))
    valid = [(m, v) for m, v in pairs if pd.notna(v)]
    return sorted(valid, key=lambda x: x[1], reverse=True)

def prettify_motif(m):
    return {
        "headgroup": "headgroup",
        "glycerol": "glycerol",
        "alkenyl": "alkenyl",
        "alkyl_chain": "alkyl-chain",
        "weak-secondary-motif": "weak secondary motif",
        "single-motif": "single motif",
        "unknown": "unknown",
    }.get(m, str(m).replace("_", "-"))

def motif_to_tag(m):
    return {
        "headgroup": "headgroup-dominant",
        "glycerol": "glycerol-dominant",
        "alkenyl": "alkenyl-dominant",
        "alkyl_chain": "alkyl-chain-dominant",
        "unknown": "motif-unknown",
    }.get(m, "motif-unknown")

def secondary_to_tag(m):
    return {
        "headgroup": "headgroup-associated",
        "glycerol": "glycerol-associated",
        "alkenyl": "alkenyl-associated",
        "alkyl_chain": "alkyl-chain-associated",
        "weak-secondary-motif": "weak-secondary-motif",
        "single-motif": "single-motif",
        "unknown": "unknown-secondary",
    }.get(m, "unknown-secondary")

def classify_second_motif(raw_ratio, soft_probs):
    ranked_soft = rank_motifs(soft_probs)
    ranked_rawr = rank_motifs(raw_ratio)

    if len(ranked_soft) < 2:
        return "single-motif"

    top1_name, top1_soft = ranked_soft[0]
    top2_name, top2_soft = ranked_soft[1]
    top2_rawr = dict(ranked_rawr).get(top2_name, np.nan)

    if pd.isna(top2_soft):
        return "single-motif"

    if (
        top2_soft >= SETTINGS["SECOND_MIN_SOFT"]
        and pd.notna(top2_rawr)
        and top2_rawr >= SETTINGS["SECOND_MIN_RAW_RATIO"]
        and top1_soft > 0
        and (top2_soft / top1_soft) >= SETTINGS["SECOND_MIN_REL_TO_TOP1"]
    ):
        return top2_name

    return "weak-secondary-motif"

def classify_scope(entropy_value):
    if pd.isna(entropy_value):
        return "motif-scope-unknown"
    if entropy_value <= SETTINGS["LOCALIZED_ENTROPY_MAX"]:
        return "localized-motif-perturbation"
    elif entropy_value <= SETTINGS["DUAL_ENTROPY_MAX"]:
        return "dual-motif-perturbation"
    else:
        return "broad-motif-perturbation"

def classify_dominance_strength(soft_probs):
    ranked = rank_motifs(soft_probs)
    if len(ranked) == 0:
        return "dominance-unknown"
    if len(ranked) == 1:
        return "strongly-dominant"

    top1 = ranked[0][1]
    top2 = ranked[1][1]
    gap = top1 - top2

    if gap < SETTINGS["AMBIGUOUS_TOP2_GAP"]:
        return "competitively-distributed"
    elif gap >= SETTINGS["STRONG_SOFT_MARGIN"]:
        return "strongly-dominant"
    elif gap >= SETTINGS["MODERATE_SOFT_MARGIN"]:
        return "moderately-dominant"
    else:
        return "weakly-dominant"

def classify_entropy_profile(entropy_value):
    if pd.isna(entropy_value):
        return "entropy-unknown"
    if entropy_value <= 0.35:
        return "low-entropy-profile"
    elif entropy_value <= 0.65:
        return "intermediate-entropy-profile"
    else:
        return "high-entropy-profile"

def motif_interpretation_sentence(dominant):
    mapping = {
        "headgroup":   "The dominant perturbed region is assigned to the lipid headgroup region.",
        "glycerol":    "The dominant perturbed region is assigned to the glycerol-associated region.",
        "alkenyl":     "The dominant perturbed region is assigned to the alkenyl-associated region.",
        "alkyl_chain": "The dominant perturbed region is assigned to the alkyl-chain region.",
        "unknown":     "No clearly resolved dominant lipid motif was identified.",
    }
    return mapping.get(dominant, "A dominant lipid-associated region was identified.")

def make_interaction_sentence(
    mode_name,
    dominant,
    second,
    scope,
    dominance_strength,
    entropy_class,
    soft_probs,
    entropy_value
):
    ranked_soft = rank_motifs(soft_probs)
    top1 = ranked_soft[0][1] if len(ranked_soft) > 0 else np.nan
    top2 = ranked_soft[1][1] if len(ranked_soft) > 1 else np.nan

    dom_txt = prettify_motif(dominant)
    sec_txt = prettify_motif(second)
    scope_txt = scope.replace("-", " ")
    dom_strength_txt = dominance_strength.replace("-", " ")
    entropy_txt = entropy_class.replace("-", " ")

    top1_txt = "NA" if pd.isna(top1) else f"{top1:.3f}"
    top2_txt = "NA" if pd.isna(top2) else f"{top2:.3f}"
    ent_txt  = "NA" if pd.isna(entropy_value) else f"{entropy_value:.3f}"

    base = motif_interpretation_sentence(dominant)

    if second in ["weak-secondary-motif", "single-motif"]:
        sec_sentence = "No robust secondary interaction motif was identified."
    else:
        sec_sentence = f"A secondary contribution is assigned to the {sec_txt} region."

    detail = (
        f"In the {mode_name}-based profile, the interaction pattern is classified as "
        f"{dom_strength_txt} with a {scope_txt} and a {entropy_txt}. "
        f"The dominant motif is {dom_txt} (soft score={top1_txt}), "
        f"the second motif is {sec_txt} (soft score={top2_txt}), "
        f"and the normalized motif entropy is {ent_txt}. "
        f"{sec_sentence}"
    )

    return base + " " + detail

def get_raw_vector(row, cols):
    vals = [safe_float(row.get(c, np.nan)) for c in cols]
    arr = np.array(vals, dtype=float)

    if np.any(np.isfinite(arr)):
        arr = np.where(np.isfinite(arr), arr, 0.0)
        arr = np.clip(arr, 0, None)
        return arr

    return np.full(len(cols), np.nan)

def build_profile(row, mode_name, score_cols, temperature):
    raw = get_raw_vector(row, score_cols)

    if np.all(~np.isfinite(raw)):
        rec = {
            f"{mode_name}_dominant_motif": "unknown",
            f"{mode_name}_second_motif": "unknown",
            f"{mode_name}_scope": "motif-scope-unknown",
            f"{mode_name}_dominance_strength": "dominance-unknown",
            f"{mode_name}_entropy_class": "entropy-unknown",
            f"{mode_name}_entropy": np.nan,
            f"{mode_name}_dominance_gap": np.nan,
            f"{mode_name}_interaction_tag": "motif-unknown",
            f"{mode_name}_interaction_sentence": "Interaction profile could not be determined.",
        }
        for m in MOTIFS:
            rec[f"{mode_name}_soft_{m}"] = np.nan
            rec[f"{mode_name}_rawratio_{m}"] = np.nan
        return rec

    raw_ratio = raw_to_ratio(raw)
    soft_probs = softmax_temperature(raw_ratio, T=temperature)

    ranked = rank_motifs(soft_probs)

    dominant = ranked[0][0] if len(ranked) > 0 else "unknown"
    top1 = ranked[0][1] if len(ranked) > 0 else np.nan
    top2 = ranked[1][1] if len(ranked) > 1 else np.nan

    second = classify_second_motif(raw_ratio, soft_probs)
    entropy_value = normalized_entropy(soft_probs)
    scope = classify_scope(entropy_value)
    dominance_strength = classify_dominance_strength(soft_probs)
    entropy_class = classify_entropy_profile(entropy_value)

    gap = top1 - top2 if pd.notna(top1) and pd.notna(top2) else np.nan

    interaction_tag = ", ".join([
        motif_to_tag(dominant),
        secondary_to_tag(second),
        scope,
        dominance_strength,
        entropy_class,
    ])

    interaction_sentence = make_interaction_sentence(
        mode_name=mode_name,
        dominant=dominant,
        second=second,
        scope=scope,
        dominance_strength=dominance_strength,
        entropy_class=entropy_class,
        soft_probs=soft_probs,
        entropy_value=entropy_value,
    )

    rec = {
        f"{mode_name}_dominant_motif": dominant,
        f"{mode_name}_second_motif": second,
        f"{mode_name}_scope": scope,
        f"{mode_name}_dominance_strength": dominance_strength,
        f"{mode_name}_entropy_class": entropy_class,
        f"{mode_name}_entropy": entropy_value,
        f"{mode_name}_dominance_gap": gap,
        f"{mode_name}_interaction_tag": interaction_tag,
        f"{mode_name}_interaction_sentence": interaction_sentence,
    }

    for m, p, rr in zip(MOTIFS, soft_probs, raw_ratio):
        rec[f"{mode_name}_soft_{m}"] = p
        rec[f"{mode_name}_rawratio_{m}"] = rr

    return rec

# Build height and AUC profiles
height_profile_rows = []
auc_profile_rows = []

for _, row in motif_df.iterrows():
    height_profile_rows.append(
        build_profile(
            row,
            mode_name="height",
            score_cols=height_cols,
            temperature=SETTINGS["SOFTMAX_TEMPERATURE_HEIGHT"],
        )
    )
    auc_profile_rows.append(
        build_profile(
            row,
            mode_name="auc",
            score_cols=auc_cols,
            temperature=SETTINGS["SOFTMAX_TEMPERATURE_AUC"],
        )
    )

height_profile_df = pd.DataFrame(height_profile_rows)
auc_profile_df = pd.DataFrame(auc_profile_rows)

df = pd.concat([motif_df.reset_index(drop=True), height_profile_df, auc_profile_df], axis=1)

# ===============================================================
# 9. Select final I profile
# ===============================================================
def choose_I_mode(row):
    h_total = safe_float(row.get("total_delta_height_score", np.nan))
    a_total = safe_float(row.get("total_delta_auc_score", np.nan))

    # AUC is generally more integrated and stable.
    # Use AUC if available; otherwise fallback to height.
    if pd.notna(a_total) and a_total > 0:
        return "auc"
    elif pd.notna(h_total) and h_total > 0:
        return "height"
    else:
        return "unknown"

df["I_selected_mode"] = df.apply(choose_I_mode, axis=1)

def select_mode_value(row, suffix):
    mode = row["I_selected_mode"]
    if mode in ["height", "auc"]:
        return row.get(f"{mode}_{suffix}", np.nan)
    return np.nan

for suffix in [
    "dominant_motif",
    "second_motif",
    "scope",
    "dominance_strength",
    "entropy_class",
    "entropy",
    "dominance_gap",
    "interaction_tag",
    "interaction_sentence",
    "soft_headgroup",
    "soft_glycerol",
    "soft_alkenyl",
    "soft_alkyl_chain",
    "rawratio_headgroup",
    "rawratio_glycerol",
    "rawratio_alkenyl",
    "rawratio_alkyl_chain",
]:
    df[f"I_{suffix}"] = df.apply(lambda r: select_mode_value(r, suffix), axis=1)

# ===============================================================
# 10. Define M information for caption
# ===============================================================
# M = motif identity / motif distribution information
# I = interaction profile / perturbation pattern information

df["M_primary_motif"] = df["I_dominant_motif"]
df["M_secondary_motif"] = df["I_second_motif"]

df["M_primary_motif_label"] = df["M_primary_motif"].apply(prettify_motif)
df["M_secondary_motif_label"] = df["M_secondary_motif"].apply(prettify_motif)

def make_M_tag(row):
    primary = row.get("M_primary_motif_label", "unknown")
    secondary = row.get("M_secondary_motif_label", "unknown")

    if secondary in ["weak secondary motif", "single motif", "unknown"]:
        return f"{primary}-centered lipid motif"
    else:
        return f"{primary}-centered lipid motif with {secondary} contribution"

df["M_motif_tag"] = df.apply(make_M_tag, axis=1)

def make_M_sentence(row):
    primary = row.get("M_primary_motif_label", "unknown")
    secondary = row.get("M_secondary_motif_label", "unknown")
    mode = row.get("I_selected_mode", "unknown")

    if secondary in ["weak secondary motif", "single motif", "unknown"]:
        return (
            f"The motif descriptor M assigns the dominant lipid-associated perturbation "
            f"to the {primary} region based on {mode}-derived suppression scores."
        )
    else:
        return (
            f"The motif descriptor M assigns the dominant lipid-associated perturbation "
            f"to the {primary} region, with an additional contribution from the "
            f"{secondary} region, based on {mode}-derived suppression scores."
        )

df["M_motif_sentence"] = df.apply(make_M_sentence, axis=1)

def make_combined_caption_fragment(row):
    cop = row.get("Copolymer_Name", "This copolymer")
    m_sentence = row.get("M_motif_sentence", "")
    i_sentence = row.get("I_interaction_sentence", "")
    return f"{cop}: {m_sentence} {i_sentence}"

df["caption_fragment_MI"] = df.apply(make_combined_caption_fragment, axis=1)

# ===============================================================
# 11. Optional merge with Copolymer_ROI_motif_scores_260312.csv
# ===============================================================
if base_df is not None and "Copolymer_Name" in base_df.columns:
    base_df["Copolymer_Name"] = base_df["Copolymer_Name"].apply(normalize_copolymer_name)

    base_keep_cols = [
        c for c in base_df.columns
        if c in [
            "Copolymer_Name",
            "headgroup_score",
            "glycerol_score",
            "alkenyl_score",
            "alkyl_chain_score",
            "headgroup_score_norm",
            "glycerol_score_norm",
            "alkenyl_score_norm",
            "alkyl_chain_score_norm",
            "dominant_motif",
            "total_motif_weighted_signal",
            "n_nonmissing_roi_values",
            "mean_abs_roi_intensity",
        ]
    ]

    base_sub = base_df[base_keep_cols].copy()
    rename_map = {
        c: f"base_{c}"
        for c in base_sub.columns
        if c != "Copolymer_Name"
    }
    base_sub = base_sub.rename(columns=rename_map)

    df = df.merge(base_sub, on="Copolymer_Name", how="left")

# ===============================================================
# 12. Useful caption-oriented summary columns
# ===============================================================
df["I_polar_soft_score"] = df["I_soft_headgroup"] + df["I_soft_glycerol"]
df["I_chain_soft_score"] = df["I_soft_alkenyl"] + df["I_soft_alkyl_chain"]
df["I_chain_minus_polar"] = df["I_chain_soft_score"] - df["I_polar_soft_score"]

def classify_polar_chain(row):
    val = row.get("I_chain_minus_polar", np.nan)
    if pd.isna(val):
        return "unknown"
    if val > 0.15:
        return "chain-associated"
    elif val < -0.15:
        return "polar-associated"
    else:
        return "balanced polar/chain-associated"

df["I_polar_chain_class"] = df.apply(classify_polar_chain, axis=1)

def make_short_caption(row):
    cop = row.get("Copolymer_Name", "")
    primary = row.get("M_primary_motif_label", "unknown")
    secondary = row.get("M_secondary_motif_label", "unknown")
    scope = str(row.get("I_scope", "unknown")).replace("-", " ")
    strength = str(row.get("I_dominance_strength", "unknown")).replace("-", " ")
    pc = row.get("I_polar_chain_class", "unknown")

    if secondary in ["weak secondary motif", "single motif", "unknown"]:
        sec_txt = "without a robust secondary motif"
    else:
        sec_txt = f"with secondary {secondary} contribution"

    return (
        f"{cop} shows a {strength} {primary}-centered M profile "
        f"{sec_txt}, corresponding to a {scope} I profile and a {pc} interaction pattern."
    )

df["caption_short_MI"] = df.apply(make_short_caption, axis=1)

# ===============================================================
# 13. Column ordering
# ===============================================================
front_cols = [
    "No",
    "Sample_Name",
    "Copolymer_Name",
    "hydrophilic_monomer",
    "hydrophobic_monomer",
    "crosslinker",

    # M information
    "M_primary_motif",
    "M_secondary_motif",
    "M_primary_motif_label",
    "M_secondary_motif_label",
    "M_motif_tag",
    "M_motif_sentence",

    # I information
    "I_selected_mode",
    "I_dominant_motif",
    "I_second_motif",
    "I_scope",
    "I_dominance_strength",
    "I_entropy_class",
    "I_entropy",
    "I_dominance_gap",
    "I_interaction_tag",
    "I_polar_soft_score",
    "I_chain_soft_score",
    "I_chain_minus_polar",
    "I_polar_chain_class",
    "I_interaction_sentence",

    # caption-ready text
    "caption_short_MI",
    "caption_fragment_MI",
]

score_cols_all = (
    height_cols
    + [c.replace("_score", "_norm") for c in height_cols]
    + auc_cols
    + [c.replace("_score", "_norm") for c in auc_cols]
    + [
        "total_delta_height_score",
        "total_delta_auc_score",
        "dominant_height_motif",
        "dominant_auc_motif",
    ]
)

soft_cols = []
for mode in ["height", "auc"]:
    soft_cols += [
        f"{mode}_dominant_motif",
        f"{mode}_second_motif",
        f"{mode}_scope",
        f"{mode}_dominance_strength",
        f"{mode}_entropy_class",
        f"{mode}_entropy",
        f"{mode}_dominance_gap",
        f"{mode}_interaction_tag",
        f"{mode}_interaction_sentence",
    ]
    for m in MOTIFS:
        soft_cols.append(f"{mode}_soft_{m}")
    for m in MOTIFS:
        soft_cols.append(f"{mode}_rawratio_{m}")

base_cols = [c for c in df.columns if c.startswith("base_")]

ordered_cols = []
for c in front_cols + score_cols_all + soft_cols + base_cols:
    if c in df.columns and c not in ordered_cols:
        ordered_cols.append(c)

remaining_cols = [c for c in df.columns if c not in ordered_cols]
df_out = df[ordered_cols + remaining_cols].copy()

# ===============================================================
# 14. Save outputs
# ===============================================================
OUT_CSV  = "/content/Copolymer_MI_caption_features.csv"
OUT_XLSX = "/content/Copolymer_MI_caption_features.xlsx"

SUMMARY_CSV  = "/content/Copolymer_MI_caption_summary.csv"
SUMMARY_XLSX = "/content/Copolymer_MI_caption_summary.xlsx"

df_out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

summary_cols = [
    c for c in front_cols
    if c in df_out.columns
]
summary_df = df_out[summary_cols].copy()
summary_df.to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as writer:
    df_out.to_excel(writer, index=False, sheet_name="MI_caption_features")
    motif_df.to_excel(writer, index=False, sheet_name="change_based_scores")
    w_df.to_excel(writer, index=False, sheet_name="functional_weights_used")

with pd.ExcelWriter(SUMMARY_XLSX, engine="xlsxwriter") as writer:
    summary_df.to_excel(writer, index=False, sheet_name="MI_caption_summary")

print("\n✅ Saved:")
print(" -", CHANGE_CSV)
print(" -", CHANGE_XLSX)
print(" -", OUT_CSV)
print(" -", OUT_XLSX)
print(" -", SUMMARY_CSV)
print(" -", SUMMARY_XLSX)

print("\nPreview:")
display(summary_df.head(10))

print("\nI dominant motif counts:")
print(df_out["I_dominant_motif"].value_counts(dropna=False))

print("\nI scope counts:")
print(df_out["I_scope"].value_counts(dropna=False))

print("\nM tag counts:")
print(df_out["M_motif_tag"].value_counts(dropna=False))

# ===============================================================
# 15. Download
# ===============================================================
files.download(OUT_CSV)
files.download(OUT_XLSX)
files.download(SUMMARY_CSV)
files.download(SUMMARY_XLSX)